# Week 2: Expected and observed outputs

Compare expected and actual outputs from each function's `all_observations.npz`. Predicted means and posterior standard deviations remain aligned to inputs and observations. The weekly JSON files are retained as tracking records.

The signed z-score is `(observed_output - expected_output) / std`: positive values indicate underprediction, and negative values indicate overprediction. Its magnitude measures error in units of predicted standard deviation. Scores are missing (`NaN`) when the standard deviation is missing or zero.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bayesian_bbo import FUNCTION_SPECS, load_observations

root = Path('../../data')
records = []

for function_id in FUNCTION_SPECS:
    function_directory = root / f'function_{function_id}'
    if not (function_directory / 'all_observations.npz').exists():
        continue
    observations = load_observations(root, function_id)
    matched = np.isfinite(observations.expected_outputs)
    for index in np.flatnonzero(matched):
        records.append({
            'function': function_id,
            'input': observations.inputs[index].tolist(),
            'expected_output': float(observations.expected_outputs[index]),
            'observed_output': float(observations.outputs[index]),
            'std': float(observations.stds[index]),
        })

if not records:
    raise ValueError('No observations have expected outputs yet.')

comparison = pd.DataFrame(records)
comparison['error'] = comparison['observed_output'] - comparison['expected_output']
comparison['absolute_error'] = comparison['error'].abs()
valid_std = comparison['std'].where(np.isfinite(comparison['std']) & (comparison['std'] > 0))
comparison['z_score'] = comparison['error'] / valid_std
comparison['absolute_z_score'] = comparison['z_score'].abs()
comparison

## Error summary

Summarize prediction error per function. Each function's outputs remain on its own scale.

In [ ]:
summary = (
    comparison.groupby('function')
    .agg(
        observations=('error', 'size'),
        mean_expected=('expected_output', 'mean'),
        mean_observed=('observed_output', 'mean'),
        mean_error=('error', 'mean'),
        max_error=('error', 'max'),
        mean_std=('std', 'mean'),
        valid_z_scores=('z_score', 'count'),
        mean_z_score=('z_score', 'mean'),
        max_absolute_z_score=('absolute_z_score', 'max'),
    )
)
summary

## Expected versus observed

Each panel uses an independent y-axis so functions with different output scales can be compared without one dominating the chart.

In [ ]:
function_ids = sorted(comparison['function'].unique())
fig, axes = plt.subplots(2, 4, figsize=(14, 8), squeeze=False)
axes = axes.ravel()

for ax, function_id in zip(axes, function_ids):
    rows = comparison[comparison['function'] == function_id]
    positions = np.arange(len(rows))
    width = 0.38
    ax.bar(positions - width / 2, rows['expected_output'], width, label='Expected')
    ax.bar(positions + width / 2, rows['observed_output'], width, label='Observed')
    ax.set_title(f'Function {function_id}')
    ax.set_xlabel('Matched input index')
    ax.set_ylabel('Output')
    ax.set_xticks(positions, [str(index + 1) for index in positions])
    ax.axhline(0, color='black', linewidth=0.7)
    ax.legend()

for ax in axes[len(function_ids):]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()